# result comparison of MCMC vs FM
This notebook is to compare the inferred posterior (conditioned on the same simulated lightcurve) resulting from MCMC sampling and a trained flow matching model 

In [19]:
import os
import sys

from corner import corner

sys.path.append('..')

from src.simulator import Model, BurstSimulator
from src.c2st import c2st
from src.flow_matching.distributions import UniformPrior, CompositePrior, Posterior
from src.flow_matching.probability_path import GuidedLinearProbabilityPath
from src.flow_matching.integration import EulerODESolver
from src.flow_matching.models import MLPGuidedVectorField, FRBLightCurveCNN, LightCurveThinner, fourier_embedding, LightCurveMLP, UNetEncoder, TransdimensionalModel, EncodedClassifier
from src.flow_matching.transformer import TransformerGuidedField
from src.helpers import record_every, plot_posterior_samples, gen_parameter_labels

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.lines as mlines

import numpy as np
import torch
import yaml

from src.flow_matching.plotting import plot_loss, plot_snapshots
from src.flow_matching.helpers import choose_device, build_mlp, find_run_dir

device = choose_device()

In [20]:
MODELS = {
    "MLPGuidedVectorField": MLPGuidedVectorField,
    "TransformerGuidedField":TransformerGuidedField
}

TIME_ENCODERS = {
    "LightCurveThinner": LightCurveThinner,
    "FRBLightCurveCNN": FRBLightCurveCNN,
    "LightCurveMLP":LightCurveMLP,
    "UNetEncoder":UNetEncoder
}

TAU_ENCODERS = {
    True : fourier_embedding,
    False : None 
}

THETA_ENCODERS = {
    # True : lambda theta_dim : build_mlp([2] + [2 * 4, 2 * 8] + [theta_dim]),
    True : lambda theta_dim : build_mlp([4] + [4 * 8, 4 * 32] + [theta_dim]),
    False : None
}

PATHS = {
    "GuidedLinearProbabilityPath": GuidedLinearProbabilityPath,
}

DISTRIBUTIONS = {
    "Posterior":     Posterior,
    "UniformPrior":  UniformPrior,
    "CompositePrior":CompositePrior,
    "NewPosterior":  Posterior
}

CLASSIFIERS = {
    "EncodedClassifier":EncodedClassifier
}

# Loading in the FM model

In [31]:
# fill in desired job_id or directory name 
job_id = "13336894" #False
run_dir = None
save_dir = "../checkpoints/"

In [32]:
# loading the model (via run_id, or path)
run_dir = find_run_dir(job_id, save_dir) if job_id else os.path.join(save_dir, run_dir)

checkpoint_path = os.path.join(run_dir, 'training_checkpoint.pth')
config_path     = os.path.join(run_dir, 'config.yaml')

In [ ]:
# create empty model from config
def read_config(path):
    with open(path, 'r') as f:
        config = yaml.safe_load(f)
    return config

def empty_model_from_config(config):
    model_name = config["model"]["name"]
    model_class = MODELS[model_name]
    
    kwargs = config["model"]["init_params"]

    time_encoder = False if not config["time_seq_encoder"] else config["time_seq_encoder"]["name"] 

    if time_encoder:
        t_encoder_kwargs = config["time_seq_encoder"]["init_params"]


    kwargs['time_seq_encoder']  = None if not time_encoder else TIME_ENCODERS[time_encoder](**t_encoder_kwargs)
    kwargs['tau_encoder']   = TAU_ENCODERS[config["tau_encoder"]]
    kwargs['theta_encoder'] = THETA_ENCODERS[config["theta_encoder"]]

    return model_class(**kwargs)

config = read_config(config_path)
vector_field = empty_model_from_config(config)

In [34]:
def empty_classifier_from_config(config):
    classifier_conf = config.get("classifier", False)
    if not classifier_conf:
        print("No classifier settings found in config")
        return False
    
    classifier_name = classifier_conf["name"]
    classifier = CLASSIFIERS[classifier_name]

    encoder = TIME_ENCODERS[classifier_conf['encoder']['name']]
    enc_kwargs = classifier_conf['encoder']['init_params']
    encoder = encoder(**enc_kwargs)
    
    cls_kwargs = classifier_conf["init_params"]
    classifier = classifier(encoder, **cls_kwargs)

    return classifier

classifier = empty_classifier_from_config(config)

In [35]:
transdimensional = True
if transdimensional:
    vector_field = TransdimensionalModel(classifier, vector_field)

In [ ]:
# load trained model 
print(checkpoint_path)
checkpoint = torch.load(checkpoint_path, weights_only=False)

# Load 'normal' or EMA version 
ema = False
if ema:
    EMA_checkpoint_path = os.path.join(run_dir, "EMA_checkpoint.pth")
    state_dict = torch.load(EMA_checkpoint_path, weights_only=False)
    vector_field.load_state_dict(state_dict) 
else:
    vector_field.load_state_dict(checkpoint["model_state_dict"])

vector_field.eval()
vector_field.to(device)

losses = checkpoint["losses"]

In [ ]:
# loading the probability path
def prob_path_from_config(config):

    path_name = config["path"]["name"]
    path_class = PATHS[path_name]
    
    inf_params = config["path"]["p_data"]["init_params"]['inf_params']
    p_simple_name = config["path"]["p_simple"]["name"]
    p_simple_cls = DISTRIBUTIONS[p_simple_name]
    kwargs = config["path"]["p_simple"]["init_params"]
    
    if p_simple_name == "CompositePrior":
        prior_dict = {}

        for param, prior in zip(inf_params, kwargs):
            prior_cls = DISTRIBUTIONS[prior['name']]
            print(prior['init_params'])
            prior_dict[param] = prior_cls(**prior['init_params'])

        p_simple = p_simple_cls(prior_dict)

    else:
        p_simple = p_simple_cls(**kwargs)
    
    p_data_name   = config["path"]["p_data"]["name"]
    p_data_cls = DISTRIBUTIONS[p_data_name]

    kwargs = config["path"]["p_data"]["init_params"]
    kwargs.pop('prior') if kwargs.get('prior', False) else None
    
    p_data = p_data_cls(prior=p_simple, **kwargs)
    path = path_class(p_simple, p_data)
    
    return path

path = prob_path_from_config(config)
inf_params = config['model']['init_params']['inf_params']
N = path.p_data.model_params['ncomp']
vector_dim = N * len(inf_params)
burstparams = path.p_data.model_params['burstparams']

try:
    mean, std = torch.tensor(config['training']['sample_mean'], device=device), torch.tensor(config['training']['sample_std'], device=device)
except KeyError:
    mean, std = torch.zeros(vector_dim, device=device), torch.ones(vector_dim, device=device)

# Loading the matching MCMC samples

In [ ]:
# check if run with same N, inf_params and burstparams exists
def mcmc_settings_exist(settings, dictionary):
    print(settings)

    for key, value in dictionary.items():

        # key does not exits
        if not settings.get(key, False):
            return False
        if key == 'burstparams':
            same_params = compare_dicts(ignore_keys=settings['inf_params'], d1=settings[key], d2=value)
        elif key == 'inf_params':
            if not set(settings[key]) == set(value):
                return False
        elif key == 'N':
            if settings[key] != value:
                return False
        
    return True 

def compare_dicts(ignore_keys, d1, d2, rel_tol=1e-9, abs_tol=0.0):
    if d1.keys() != d2.keys():
        return False
    return np.all(np.isclose(d1[k], d2[k], rel_tol=rel_tol, abs_tol=abs_tol) for k in d1 if k not in ignore_keys)

def mcmc_run_exists(N, inf_params, burstparams, save_dir='../MCMC_runs'):
    dirs = os.listdir(save_dir)

    for dir in dirs:

        with open(os.path.join(save_dir, dir, 'settings.yaml'), 'r') as f:
            settings = yaml.safe_load(f)

        if mcmc_settings_exist(settings, {'N':N, 'inf_params':inf_params, 'burstparams':burstparams}):
            return os.path.join(save_dir, dir)
        
    return False

run_path = mcmc_run_exists(N, inf_params, burstparams)
if not run_path:
    print("No matching MCMC run found")
else:
    print(f'matching mcmc run found at {run_path}!')
    settings_file = open(os.path.join(run_path, 'settings.yaml'), 'r')
    burstparams = yaml.safe_load(settings_file)['burstparams']
    settings_file.close()

    samples = np.load(os.path.join(run_path, 'samples.npy'))
    simulated_counts = np.load(os.path.join(run_path, 'simulated_counts.npy'))

In [ ]:
plt.plot(simulated_counts)

# generate FM posterior 

In [ ]:
num_samples = 25000  # number of prior samples to transform 
samples_per_batch = 4096
batches = num_samples // samples_per_batch
final_snapshot = torch.zeros((batches * samples_per_batch, vector_dim), device=device)
Ns = torch.zeros((batches * samples_per_batch, 1), device=device)

# use same data point for conditioning all prior samples
simulations = torch.tensor(simulated_counts, device=device, dtype=torch.float).repeat(samples_per_batch, 1)

# initialize ODE solver
solver = EulerODESolver(vector_field)
nts = 200
ts = torch.linspace(0, 1, nts).to(device)

# integrate in batches
for i in range(batches):
    # simulate ODE starting from x0
    x0 = (path.p_simple.sample(samples_per_batch).to(device) - mean) / std

    start = i * samples_per_batch
    stop = start + samples_per_batch
    
    logits = classifier(simulations)
    
    N_samples = torch.multinomial(torch.softmax(logits, dim=1), num_samples=1) + 1
    Ns[start:stop, :] = N_samples
    final_snapshot[start:stop, :] = solver.solve(x0, ts.view(1, nts, 1).expand(samples_per_batch, nts, 1), y=simulations, N=N_samples) * std + mean

# Corner plot overlay

In [ ]:
range_ = [1, 1]

N_inf = N # choose for which N to make corner plot (for FM)
FM_samples = final_snapshot[(Ns == N_inf).view(Ns.size(0))]

data = [FM_samples.cpu().numpy(), samples]
colors=['blue', 'red']
labels=['FM', 'MCMC']

var_names = gen_parameter_labels(inf_params, N)
true_values = np.array([burstparams[key] for key in inf_params]).flatten()

fig = None
for i in range(1):#len(data)):
    fig = corner(data[i], labels=var_names, truths=true_values, range=[range_[i] for _ in range(N * len(inf_params))], 
             color=colors[i], truth_color='black', label=labels[i], fig=fig, plot_density=False, plot_datapoints=False, 
             fill_contours=True, plot_contours=True, hist_kwargs={'density':True}, bins=20)

# make legend with dummy lines
plt.legend(
        handles=[
            mlines.Line2D([], [], color=colors[i], label=labels[i])
            for i in range(len(data))
        ],
        fontsize=20, frameon=False,
        bbox_to_anchor=(1, 2), loc="upper right"
    )
plt.show()


# Posterior samples

In [ ]:
modelparams = {'time':np.linspace(0, 1, 1000), 'burstparams':burstparams, 'ybkg':5,'ncomp':N}
true_flux = Model(**modelparams).get_flux()

plt.figure(figsize=(12,5))

for i in range(2):

    plt.subplot(121 + i)

    plot_posterior_samples(
        100, 
        simulated_counts, 
        data[i], 
        inf_params, 
        modelparams={'time':np.linspace(0, 1, 1000), 'burstparams':burstparams, 'ybkg':5,'ncomp':3},
        true_flux=true_flux,
        title=f"100 {labels[i]} posterior samples"
        )

# C2ST

Performs classifier two-sample test. A binary classifier is trained to distinguish samples from MCMC and FM, and its accuracy is evaluated. If the accuracy is 50\%, this indicates the classifier is unable to distinguish between the two distributions. If the accuracy is 100\%, the classifier can easily distinguish between samples from MCMC and FM. I.e. lower score is better and the 'best' minimum score is 50%. 

In [ ]:
FM_samples = final_snapshot 

# ensure number of MCMC samples equal to FM
MCMC_samples = np.copy(samples)
np.random.shuffle(MCMC_samples)
N, _ = FM_samples.shape
MCMC_samples = torch.tensor(MCMC_samples[:N], device=device, dtype=torch.float32)

# standardize using common mean and std
concat = torch.cat((FM_samples, MCMC_samples))
mean  = torch.mean(concat)
std  = torch.std(concat)

MCMC_samples = (MCMC_samples - mean) / std 
FM_samples   = (FM_samples - mean) / std 

accuracy = c2st(FM_samples, MCMC_samples)

In [ ]:
print(f'c2st score: {accuracy}')